# COAD Tumor vs Normal Predictor - Full Workflow / 完整流程

This notebook combines the original step notebooks into one runnable workflow and now uses local TCGA COAD tumor samples plus UCSC Xena Toil GTEx normal colon samples.

这个 notebook 把原来的分步骤文件合并成一个完整流程：准备数据、训练模型、查看指标、解释重要基因、生成/查看报告。当前版本使用本地 TCGA COAD tumor 样本，加上 UCSC Xena Toil GTEx normal colon 样本。

**Research-only caveat / 重要限制：** this project predicts COAD tumor vs normal from RNA expression data for research exploration only. Tumor data comes from local TCGA PanCan Atlas tables, while normal data comes from UCSC Xena Toil GTEx colon tissue. Results may include cohort/source effects (differences caused by data source or processing / 由数据来源或处理方法造成的差异), not only biology.


## 0. Setup / 设置路径

Run this first so imports work whether the notebook kernel starts inside the project root or inside the `notebooks/` folder.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROJECT_ROOT


## 1. Prepare COAD Data / 准备 COAD 数据

Build the COAD tumor/normal RNA expression feature matrix from PostgreSQL using TCGA COAD tumor and GTEx normal colon samples.

从 PostgreSQL 数据库读取 TCGA COAD tumor 和 GTEx normal colon RNA expression 数据，并生成后续模型需要的特征矩阵。


In [ ]:
from src.pipeline import prepare

prepare()


## 1.1 Class Imbalance Check / 类别不平衡检查

Class imbalance means one label has many more samples than another label (类别不平衡：某一类样本数量远多于另一类). After adding GTEx normal colon samples, the dataset is much more balanced than before, but accuracy alone can still be misleading.

类别不平衡会影响预测：加入 GTEx normal 后 normal 样本更多了，但仍然要看 `balanced_accuracy`、`normal_recall` 和 confusion matrix（混淆矩阵：真实类别和预测类别的对照表）。


In [ ]:
summary_path = DATA_DIR / "data_summary.json"
labels_path = DATA_DIR / "coad_tumor_normal_labels.csv"

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    tumor_count = int(summary.get("tumor_samples", 0))
    normal_count = int(summary.get("normal_samples", 0))
    ratio = tumor_count / normal_count if normal_count else None
    display(pd.DataFrame([
        {"label": "tumor", "samples": tumor_count},
        {"label": "normal", "samples": normal_count},
    ]))
    print(f"Tumor:normal ratio = {ratio:.2f}:1" if ratio else "Normal sample count is 0.")
elif labels_path.exists():
    labels = pd.read_csv(labels_path)
    display(labels["label"].value_counts().rename_axis("label").reset_index(name="samples"))
else:
    print("Run the prepare step first, then rerun this cell.")


## 2. Train Baseline Models / 训练基础模型

Train Logistic Regression, Random Forest, and Linear SVM baselines on the updated TCGA tumor + GTEx normal dataset.

训练三个基础模型：Logistic Regression（逻辑回归）, Random Forest（随机森林）, and Linear SVM（线性支持向量机）。The current training code still uses `class_weight="balanced"`, so the model remains cautious about label imbalance.


In [ ]:
from src.pipeline import train_and_report

train_and_report()


## 3. Evaluate Models / 查看模型指标

Read generated metrics after `train_and_report()` runs. Focus on `balanced_accuracy`, per-class recall, ROC-AUC, and PR-AUC, not only `accuracy`.


In [ ]:
metrics = pd.read_csv(REPORTS_DIR / "metrics_summary.csv")
display(metrics)


## 4. Interpret Important Genes / 查看重要基因

Inspect important genes generated by the baseline models. These are candidate signals, not confirmed biological conclusions.


In [ ]:
important_genes = pd.read_csv(REPORTS_DIR / "important_genes.csv")
display(important_genes.head(30))


## 5. Xena Toil Extension / Xena Toil 扩展

The GTEx normal colon extension has now been loaded locally and is used by this workflow. A stronger future version should also use Toil-reprocessed TCGA tumor data to reduce source effects.

GTEx normal colon 扩展已经加载到本地数据库，并被当前流程使用。下一版更理想的做法是同时使用 Toil 重新处理过的 TCGA tumor，进一步减少 source effects。


## 6. Model Report / 模型报告

The cells below come from the original `06_model_report.ipynb` and provide a more detailed model-report view.


## 06 COAD Model Report / 模型查看报告

这个 notebook 用来查看已经训练好的 COAD tumor vs normal RNA expression 模型。它不会修改数据库，也不会重新训练模型，只读取 `data/`、`models/`、`reports/` 里的结果。

**Research-only caveat / 重要限制：** tumor data comes from local TCGA PanCan Atlas tables (`bio_tcga`), while normal data comes from UCSC Xena Toil GTEx colon samples (`toil_gtex_colon_normal`). 两组数据来自不同 cohort/source，所以结果只适合科研探索和报告展示，不能当作临床诊断模型。

### 1. Load Report Artifacts / 读取报告产物

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

summary = json.loads((DATA_DIR / "data_summary.json").read_text(encoding="utf-8"))
metrics = pd.read_csv(REPORTS_DIR / "metrics_summary.csv")
important_genes = pd.read_csv(REPORTS_DIR / "important_genes.csv")
features = pd.read_parquet(DATA_DIR / "coad_tumor_normal_features.parquet")
labels = pd.read_csv(DATA_DIR / "coad_tumor_normal_labels.csv")
test_samples = pd.read_csv(DATA_DIR / "test_samples.csv")

display(Markdown(
    f"""
### Dataset Summary / 数据摘要

- Tumor samples: **{summary['tumor_samples']}**
- Normal samples: **{summary['normal_samples']}**
- Shared genes before low-variance filtering: **{summary['shared_genes']}**
- Model feature matrix shape: **{features.shape[0]} samples x {features.shape[1]} genes**
- Tumor source: `{summary['tumor_source']}`
- Normal source: `{summary['normal_source']}`
    """
))

### 2. Metrics / 模型效果指标

`accuracy` 看总体正确率；`balanced_accuracy` 更适合当前这种 normal 样本比较少的情况；`ROC-AUC` 和 `PR-AUC` 用来观察分类区分能力。

In [ ]:
metric_cols = [
    "model", "accuracy", "balanced_accuracy", "f1", "roc_auc", "pr_auc",
    "normal_precision", "normal_recall", "tumor_precision", "tumor_recall",
]
metrics[metric_cols].style.format({
    "accuracy": "{:.4f}",
    "balanced_accuracy": "{:.4f}",
    "f1": "{:.4f}",
    "roc_auc": "{:.4f}",
    "pr_auc": "{:.4f}",
    "normal_precision": "{:.4f}",
    "normal_recall": "{:.4f}",
    "tumor_precision": "{:.4f}",
    "tumor_recall": "{:.4f}",
})

### 3. Plots / 图表

下面三张图分别是 confusion matrix、ROC curve、PR curve。科研报告里建议配合 caveat 一起解释，不要只展示接近完美的指标。

In [ ]:
for filename in ["confusion_matrix.png", "roc_curve.png", "pr_curve.png"]:
    display(Markdown(f"### {filename}"))
    display(Image(filename=str(REPORTS_DIR / filename)))

### 4. What Is Inside The Saved Models? / 模型文件里有什么

每个 `.joblib` 文件保存了一个 sklearn `Pipeline`，里面包括 preprocessing step 和 model step。预测新样本时必须使用同一个 Pipeline，不能只拿最后的 classifier。

In [ ]:
model_rows = []
loaded_models = {}
for path in sorted(MODELS_DIR.glob("*.joblib")):
    payload = joblib.load(path)
    pipeline = payload["pipeline"]
    loaded_models[payload["model_name"]] = payload
    model_rows.append({
        "model": payload["model_name"],
        "file": path.name,
        "pipeline_steps": " -> ".join(pipeline.named_steps.keys()),
        "classifier": pipeline.named_steps["model"].__class__.__name__,
        "n_features": len(payload["features"]),
        "random_state": payload["random_state"],
    })

pd.DataFrame(model_rows)

### 5. Example Predictions / 看模型如何预测测试样本

`prediction` 是模型预测标签；`score_or_probability` 对 Logistic Regression 和 Random Forest 是 tumor probability，对 Linear SVM 是 decision score。

In [ ]:
classes = np.array(["normal", "tumor"])
test_features = features.loc[test_samples["sample_id"]]
preview = test_samples[["sample_id", "label", "source_schema", "sample_type"]].copy()

for name, payload in loaded_models.items():
    pipeline = payload["pipeline"]
    pred = pipeline.predict(test_features)
    preview[f"{name}_prediction"] = classes[pred]
    if hasattr(pipeline, "predict_proba"):
        preview[f"{name}_score_or_probability"] = pipeline.predict_proba(test_features)[:, 1]
    else:
        preview[f"{name}_score_or_probability"] = pipeline.decision_function(test_features)

preview.head(12)

### 6. Important Genes / 重要基因候选

这些 gene 是模型用来区分 tumor/normal 的 predictive features，不等于已经证明的癌症 driver genes。后续写科研报告时，需要做 literature review。

In [ ]:
important_genes.groupby("model").head(15)[[
    "model", "rank", "gene_symbol", "importance_score", "direction", "short_explanation"
]]

### 7. How To Explain This Model / 应该怎么看这个模型

- 第一眼看 `balanced_accuracy`、`normal_recall` 和 confusion matrix。加入 GTEx 后 normal 样本增加到 308 个，类别更平衡，但仍不能只看 accuracy。
- 如果 Random Forest 结果是 1.0，不要直接说模型完美；更合理的解释是：当前数据来源差异可能让任务变得过于容易，需要外部验证。
- Logistic Regression 和 Linear SVM 的 coefficients 更适合做报告解释：positive coefficient means higher tumor score，negative coefficient means higher normal score。
- `important_genes.csv` 可以作为下一步查文献的候选列表，不要直接写成因果结论。
- 真正发表或严肃展示前，下一步应该用同一套 Toil-reprocessed TCGA tumor + GTEx normal 数据重新验证，检查模型是不是学到了 cancer biology，而不是 source/cohort effect。